In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/thesis_rag'
!pip install -q -r "{PROJECT_ROOT}/config/requirements-core.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8

In [ ]:
import os

SEED = 42
N_EVAL, N_TUNING, N_RAGAS = 500, 50, 150

PROJECT_ROOT = '/content/drive/MyDrive/thesis_rag'
SQUAD_DIR = os.path.join(PROJECT_ROOT, 'data', 'squad')
os.makedirs(SQUAD_DIR, exist_ok=True)

In [ ]:
from datasets import load_dataset

squad = load_dataset('rajpurkar/squad', split='validation')
print(squad)
print('\nExample row:')
print(squad[0])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10570
})

Example row:
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represen

In [ ]:
context_to_id = {}
corpus = []

for row in squad:
    ctx = row['context']
    if ctx not in context_to_id:
        doc_id = f'squad_{len(corpus):05d}'
        context_to_id[ctx] = doc_id
        corpus.append({'doc_id': doc_id, 'title': row['title'], 'text': ctx})

print('Unique context paragraphs (corpus size):', len(corpus))

Unique context paragraphs (corpus size): 2067


In [ ]:
questions = []
dropped = 0

for row in squad:
    ans_texts = row['answers']['text']
    ans_starts = row['answers']['answer_start']
    if not ans_texts:
        dropped += 1
        continue

    ctx = row['context']
    primary = ans_texts[0]
    start = ans_starts[0]

    # Verify the offset points at the answer; if not, relocate by search.
    if ctx[start:start + len(primary)] != primary:
        found = ctx.find(primary)
        if found == -1:
            dropped += 1
            continue
        start = found

    questions.append({
        'qid': row['id'],
        'question': row['question'],
        'doc_id': context_to_id[ctx],
        'answers': ans_texts,          # keep all variants for later use
        'answer_start': start,         # verified offset into the gold context
    })

print('Usable questions:', len(questions), '| dropped:', dropped)

Usable questions: 10570 | dropped: 0


In [ ]:
import random

rng = random.Random(SEED)
pool = questions[:]
rng.shuffle(pool)

assert len(pool) >= N_EVAL + N_TUNING, 'Question pool too small for requested splits'

eval_set = pool[:N_EVAL]
tuning_set = pool[N_EVAL:N_EVAL + N_TUNING]
ragas_set = eval_set[:N_RAGAS]   # nested subset of eval, not a separate draw

print('eval:', len(eval_set), '| tuning:', len(tuning_set), '| ragas:', len(ragas_set))

# Sanity: eval and tuning must not share any question id
assert not ({q['qid'] for q in eval_set} & {q['qid'] for q in tuning_set})
print('Confirmed: eval and tuning are disjoint.')

eval: 500 | tuning: 50 | ragas: 150
Confirmed: eval and tuning are disjoint.


In [ ]:
import json, datetime

def save_json(obj, name):
    path = os.path.join(SQUAD_DIR, name)
    with open(path, 'w') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    n = len(obj) if isinstance(obj, list) else '-'
    print(f'saved {name}  ({n} records)')

save_json(corpus, 'corpus.json')
save_json(eval_set, 'squad_eval.json')
save_json(tuning_set, 'squad_tuning.json')
save_json(ragas_set, 'squad_ragas.json')

manifest = {
    'dataset': 'rajpurkar/squad (v1.1)',
    'split': 'validation',
    'seed': SEED,
    'counts': {
        'corpus_documents': len(corpus),
        'eval': len(eval_set),
        'tuning': len(tuning_set),
        'ragas': len(ragas_set),
        'dropped_questions': dropped,
    },
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
}
save_json(manifest, 'squad_manifest.json')

saved corpus.json  (2067 records)
saved squad_eval.json  (500 records)
saved squad_tuning.json  (50 records)
saved squad_ragas.json  (150 records)
saved squad_manifest.json  (- records)


In [ ]:
q = eval_set[0]
ctx = next(d['text'] for d in corpus if d['doc_id'] == q['doc_id'])
s, a = q['answer_start'], q['answers'][0]

print('Question:', q['question'])
print('Answer  :', a)
print('Gold doc:', q['doc_id'])
print('Context around the answer:')
print('  ...', repr(ctx[max(0, s - 60): s + len(a) + 60]), '...')

Question: How does the secondary theory say most cpDNA is structured?
Answer  : linear
Gold doc: squad_01693
Context around the answer:
  ... 'wever, a second theory suggests that most cpDNA is actually linear and replicates through homologous recombination. It further' ...


MMLU part

In [ ]:
from datasets import load_dataset
from collections import defaultdict
import random, json, os, datetime

SEED = 42
N_EVAL, N_TUNING, N_RAGAS = 500, 50, 150

# Knowledge-heavy subjects only: ones where retrieved factual text can help.
# Reasoning-heavy subjects (maths, logic) are deliberately excluded.
SUBJECTS = sorted([
    'high_school_world_history', 'high_school_us_history', 'prehistory',
    'world_religions', 'clinical_knowledge', 'anatomy', 'medical_genetics',
    'nutrition', 'high_school_biology', 'college_biology',
    'high_school_geography', 'sociology', 'global_facts',
    'jurisprudence', 'astronomy',
])
print(len(SUBJECTS), 'subjects selected')

15 subjects selected


In [ ]:
mmlu = load_dataset('cais/mmlu', 'all', split='test')
print(mmlu)

# Keep only our subjects; build a clean record per question.
by_subject = defaultdict(list)
for row in mmlu:
    subj = row['subject']
    if subj not in SUBJECTS:
        continue
    recs = by_subject[subj]
    ans = row['answer']
    choices = row['choices']
    recs.append({
        'qid': f'mmlu_{subj}_{len(recs):04d}',
        'subject': subj,
        'question': row['question'],
        'choices': choices,
        'answer_idx': ans,
        'answer_text': choices[ans],
    })

print('\nQuestions available per subject:')
for s in SUBJECTS:
    print(f'  {s:32s} {len(by_subject[s])}')

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 14042
})

Questions available per subject:
  anatomy                          135
  astronomy                        152
  clinical_knowledge               265
  college_biology                  144
  global_facts                     100
  high_school_biology              310
  high_school_geography            198
  high_school_us_history           204
  high_school_world_history        237
  jurisprudence                    108
  medical_genetics                 100
  nutrition                        306
  prehistory                       324
  sociology                        201
  world_religions                  171


In [ ]:
def distribute(total, k):
    # Split `total` into k near-equal integers summing exactly to total.
    base, rem = divmod(total, k)
    return [base + (1 if i < rem else 0) for i in range(k)]

eval_q = distribute(N_EVAL, len(SUBJECTS))
tune_q = distribute(N_TUNING, len(SUBJECTS))
rag_q  = distribute(N_RAGAS, len(SUBJECTS))

rng = random.Random(SEED)
eval_set, tuning_set, ragas_set = [], [], []

for i, subj in enumerate(SUBJECTS):
    items = by_subject[subj][:]
    rng.shuffle(items)

    need = eval_q[i] + tune_q[i]
    assert len(items) >= need, f'{subj}: only {len(items)} available, need {need}'

    e = items[:eval_q[i]]
    t = items[eval_q[i]:eval_q[i] + tune_q[i]]
    r = e[:rag_q[i]]
    eval_set += e
    tuning_set += t
    ragas_set += r

print('eval:', len(eval_set), '| tuning:', len(tuning_set), '| ragas:', len(ragas_set))

assert not ({q['qid'] for q in eval_set} & {q['qid'] for q in tuning_set})
print('Confirmed: eval and tuning are disjoint.')

eval: 500 | tuning: 50 | ragas: 150
Confirmed: eval and tuning are disjoint.


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/thesis_rag'
MMLU_DIR = os.path.join(PROJECT_ROOT, 'data', 'mmlu')
os.makedirs(MMLU_DIR, exist_ok=True)

def save_json(obj, name):
    path = os.path.join(MMLU_DIR, name)
    with open(path, 'w') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    n = len(obj) if isinstance(obj, list) else '-'
    print(f'saved {name}  ({n} records)')

save_json(eval_set, 'mmlu_eval.json')
save_json(tuning_set, 'mmlu_tuning.json')
save_json(ragas_set, 'mmlu_ragas.json')

manifest = {
    'dataset': 'cais/mmlu (all, test split)',
    'seed': SEED,
    'subjects': SUBJECTS,
    'counts': {'eval': len(eval_set), 'tuning': len(tuning_set), 'ragas': len(ragas_set)},
    'per_subject_eval': {SUBJECTS[i]: eval_q[i] for i in range(len(SUBJECTS))},
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
}
save_json(manifest, 'mmlu_manifest.json')

saved mmlu_eval.json  (500 records)
saved mmlu_tuning.json  (50 records)
saved mmlu_ragas.json  (150 records)
saved mmlu_manifest.json  (- records)


In [ ]:
q = eval_set[234]
print('Subject:', q['subject'])
print('Q:', q['question'])
for i, c in enumerate(q['choices']):
    mark = '  <-- correct' if i == q['answer_idx'] else ''
    print(f'   {chr(65 + i)}. {c}{mark}')

Subject: high_school_geography
Q: Which one of the following is NOT an advantage of urban agriculture?
   A. Helping to solve the problem of solid waste disposal
   B. Fresh produce for sale to others
   C. Beautification of a dingy urban area
   D. Renewed or purified water supplies  <-- correct


In [ ]:
import os, json, time, datetime, statistics

PROJECT_ROOT = '/content/drive/MyDrive/thesis_rag'
MMLU_DIR = os.path.join(PROJECT_ROOT, 'data', 'mmlu')

# Adjustable corpus-building parameters
RESULTS_PER_Q = 2      # top-N Wikipedia articles kept per question
CHAR_CAP = 8000        # max characters kept per article
SLEEP = 1.0            # ~1 request/sec: the safe rate for the Wikipedia API

def load_json(name):
    with open(os.path.join(MMLU_DIR, name)) as f:
        return json.load(f)

questions = load_json('mmlu_eval.json') + load_json('mmlu_tuning.json')
print('questions to cover:', len(questions))

questions to cover: 550


In [ ]:
import requests

WIKI_API = 'https://en.wikipedia.org/w/api.php'

session = requests.Session()
session.headers.update({
    'User-Agent': 'thesis-rag-corpus/1.0 (Masters research; duttasowmily@gmail.com)'
})

def wiki_get(params, max_retries=6):
    """One robust API call: honours 429/503 backoff and maxlag."""
    params = {**params, 'format': 'json', 'maxlag': 5}
    delay = 1.0
    for _ in range(max_retries):
        r = session.get(WIKI_API, params=params, timeout=30)
        if r.status_code == 200:
            data = r.json()
            if isinstance(data, dict) and data.get('error', {}).get('code') == 'maxlag':
                time.sleep(delay); delay *= 2; continue
            return data
        if r.status_code in (429, 503):
            wait = int(r.headers.get('Retry-After', delay))
            time.sleep(max(wait, delay)); delay *= 2; continue
        r.raise_for_status()
    raise RuntimeError(f'gave up after {max_retries} retries')

def wiki_search(query, limit=RESULTS_PER_Q):
    data = wiki_get({'action': 'query', 'list': 'search',
                     'srsearch': query, 'srlimit': limit, 'srprop': ''})
    return [h['title'] for h in data.get('query', {}).get('search', [])]

def wiki_extract(title):
    data = wiki_get({'action': 'query', 'prop': 'extracts',
                     'explaintext': 1, 'redirects': 1, 'titles': title})
    pages = data.get('query', {}).get('pages', {})
    for _, page in pages.items():
        return page.get('extract', '') or ''
    return ''

In [ ]:
search_cache_path = os.path.join(MMLU_DIR, 'wiki_search_cache.json')
search_cache = json.load(open(search_cache_path)) if os.path.exists(search_cache_path) else {}

def build_query(q):
    return f"{q['question']} {q['answer_text']}"

new = 0
for i, q in enumerate(questions):
    if q['qid'] in search_cache:
        continue
    try:
        search_cache[q['qid']] = wiki_search(build_query(q))
        new += 1
    except Exception as e:
        print('search failed', q['qid'], str(e)[:60])
    time.sleep(SLEEP)
    if (i + 1) % 50 == 0:
        json.dump(search_cache, open(search_cache_path, 'w'))
        print(f'  progress {i + 1}/{len(questions)}  (new this run: {new})')

json.dump(search_cache, open(search_cache_path, 'w'))
all_titles = sorted({t for titles in search_cache.values() for t in titles})
print('searches cached:', len(search_cache), '| unique titles:', len(all_titles))

searches cached: 550 | unique titles: 546


In [ ]:
extract_cache_path = os.path.join(MMLU_DIR, 'wiki_extract_cache.json')
extract_cache = json.load(open(extract_cache_path)) if os.path.exists(extract_cache_path) else {}

new = 0
for i, title in enumerate(all_titles):
    if title in extract_cache:            # already fetched on an earlier run
        continue
    try:
        extract_cache[title] = wiki_extract(title)[:CHAR_CAP]
        new += 1
    except Exception as e:
        print('fetch failed', title, str(e)[:60])       # NOT cached; retried next run
    time.sleep(SLEEP)
    if (i + 1) % 50 == 0:
        json.dump(extract_cache, open(extract_cache_path, 'w'))
        print(f'  progress {i + 1}/{len(all_titles)}  (new this run: {new})')

json.dump(extract_cache, open(extract_cache_path, 'w'))
print('articles fetched:', len(extract_cache))

articles fetched: 546


In [ ]:
# Which subjects' questions led to each article (metadata for later analysis)
title_subjects = {}
for q in questions:
    for t in search_cache.get(q['qid'], []):
        title_subjects.setdefault(t, set()).add(q['subject'])

corpus, skipped = [], 0
for title in all_titles:
    text = extract_cache.get(title, '').strip()
    if len(text) < 200:                   # drop empty pages and stubs
        skipped += 1
        continue
    corpus.append({
        'doc_id': f'wiki_{len(corpus):05d}',
        'title': title,
        'subjects': sorted(title_subjects.get(title, [])),
        'text': text,
    })

lengths = [len(d['text']) for d in corpus]
print('corpus documents:', len(corpus), '| skipped short/empty:', skipped)
print('total characters:', sum(lengths))
print('avg chars/article:', int(statistics.mean(lengths)) if lengths else 0)
print('rough chunk estimate @ ~800 chars:', sum(lengths) // 800)

json.dump(corpus, open(os.path.join(MMLU_DIR, 'corpus.json'), 'w'),
          ensure_ascii=False, indent=2)

manifest = {
    'source': 'English Wikipedia via MediaWiki API',
    'built_from': 'mmlu eval + tuning questions',
    'results_per_question': RESULTS_PER_Q,
    'char_cap': CHAR_CAP,
    'documents': len(corpus),
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
}
json.dump(manifest, open(os.path.join(MMLU_DIR, 'corpus_manifest.json'), 'w'), indent=2)
print('saved corpus.json and corpus_manifest.json')

corpus documents: 546 | skipped short/empty: 0
total characters: 4292994
avg chars/article: 7862
rough chunk estimate @ ~800 chars: 5366
saved corpus.json and corpus_manifest.json


In [ ]:
d = corpus[9]
print('Title:', d['title'])
print('Subjects that fetched it:', d['subjects'])
print('First 400 chars:\n', d['text'][:400])

Title: Agriculture in ancient Rome
Subjects that fetched it: ['high_school_geography']
First 400 chars:
 Roman agriculture describes the farming practices of ancient Rome, during a period of over 1000 years. From humble beginnings, the Roman Republic (509 BC–27 BC) and the Roman Empire (27 BC–476 AD) expanded to rule much of Europe, northern Africa, and the Middle East and thus comprised many agricultural environments of which the Mediterranean climate of dry, hot summers and cool, rainy winters was 


In [ ]:
import statistics
lengths = [len(d['text']) for d in corpus]
capped = sum(1 for L in lengths if L >= CHAR_CAP - 5)
print('shortest:', min(lengths), '| longest:', max(lengths))
print('median:', int(statistics.median(lengths)))
print(f'articles hitting the cap: {capped} of {len(lengths)} ({100*capped//len(lengths)}%)')
print('\nSample titles:')
for d in corpus[:15]:
    print(' -', d['title'])

shortest: 485 | longest: 8000
median: 8000
articles hitting the cap: 520 of 546 (95%)

Sample titles:
 - 14th Dalai Lama
 - 2012 in science
 - 2016 Indian banknote demonetisation
 - 2020s anti-LGBTQ movement in the United States
 - 44th Medical Brigade
 - AMP-activated protein kinase
 - Actin
 - Adenosine triphosphate
 - Agriculture in Mexico
 - Agriculture in ancient Rome
 - Air conditioning
 - Akal Sena
 - Alpha motor neuron
 - Alzheimer's disease
 - American cuisine


In [ ]:
import os, json

PROJECT_ROOT = '/content/drive/MyDrive/thesis_rag'
EMBED_MODEL = 'BAAI/bge-base-en-v1.5'
MAX_MODEL_TOKENS = 512

os.makedirs(os.path.join(PROJECT_ROOT, 'config'), exist_ok=True)
json.dump(
    {'embedding_model': EMBED_MODEL, 'max_seq_length': MAX_MODEL_TOKENS, 'dim': 768},
    open(os.path.join(PROJECT_ROOT, 'config', 'embedding_model.json'), 'w'), indent=2)

# Verify it loads on the T4 and returns 768-dim vectors
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed = HuggingFaceEmbedding(model_name=EMBED_MODEL, device='cuda')
print('embedding dim:', len(embed.get_text_embedding('a quick verification sentence')))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dim: 768


In [ ]:
import logging
import numpy as np
from transformers import AutoTokenizer

logging.getLogger('transformers.tokenization_utils_base').setLevel(logging.ERROR)

tok = AutoTokenizer.from_pretrained(EMBED_MODEL)

def load_corpus(rel):
    return json.load(open(os.path.join(PROJECT_ROOT, rel)))

squad_corpus = load_corpus('data/squad/corpus.json')
mmlu_corpus  = load_corpus('data/mmlu/corpus.json')

def token_lengths(corpus):
    return np.array([len(tok(d['text'], add_special_tokens=False,
                             truncation=False)['input_ids']) for d in corpus])

squad_len = token_lengths(squad_corpus)
mmlu_len  = token_lengths(mmlu_corpus)
print('SQuAD docs:', len(squad_len), '| MMLU docs:', len(mmlu_len))

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

SQuAD docs: 2067 | MMLU docs: 546


In [ ]:
def describe(name, lens):
    p = np.percentile(lens, [25, 50, 75, 90, 95, 99])
    print(f'\n{name}  (n={len(lens)})')
    print(f'  min {lens.min():5d} | mean {lens.mean():6.0f} | max {lens.max():5d}')
    print(f'  p25 {p[0]:.0f} | p50 {p[1]:.0f} | p75 {p[2]:.0f} | '
          f'p90 {p[3]:.0f} | p95 {p[4]:.0f} | p99 {p[5]:.0f}')

describe('SQuAD paragraphs', squad_len)
describe('MMLU articles', mmlu_len)


SQuAD paragraphs  (n=2067)
  min    30 | mean    160 | max   789
  p25 117 | p50 146 | p75 186 | p90 241 | p95 278 | p99 419

MMLU articles  (n=546)
  min   152 | mean   1668 | max  2482
  p25 1580 | p50 1678 | p75 1794 | p90 1882 | p95 1957 | p99 2105


In [ ]:
import math

def split_analysis(name, lens):
    print(f'\n{name}')
    print(f'  {"size":>5} {"% docs split":>13} {"avg chunks/doc":>16} {"total chunks":>14}')
    for S in (128, 256, 512):
        split_pct = 100 * np.mean(lens > S)
        chunks = sum(math.ceil(L / S) if L > 0 else 1 for L in lens)
        print(f'  {S:5d} {split_pct:12.0f}% {chunks/len(lens):16.1f} {chunks:14d}')

split_analysis('SQuAD paragraphs', squad_len)
split_analysis('MMLU articles', mmlu_len)


SQuAD paragraphs
   size  % docs split   avg chunks/doc   total chunks
    128           64%              1.7           3589
    256            8%              1.1           2234
    512            0%              1.0           2076

MMLU articles
   size  % docs split   avg chunks/doc   total chunks
    128          100%             13.5           7392
    256          100%              7.0           3837
    512           99%              3.8           2081


In [ ]:
import os, json
PROJECT_ROOT = '/content/drive/MyDrive/thesis_rag'
chunk_config = {
    'chunk_size_tokens': 128,
    'chunk_overlap_tokens': 32,
    'tokenizer': 'BAAI/bge-base-en-v1.5',
    'applies_to': ['fixed', 'overlapping'],
    'note': 'semantic chunking size is set by threshold tuning (Phase 3.3), '
            'targeted to a comparable average so size is not confounded with strategy',
}
json.dump(chunk_config, open(os.path.join(PROJECT_ROOT, 'config', 'chunk_config.json'), 'w'), indent=2)
print('saved chunk_config.json:', chunk_config)

saved chunk_config.json: {'chunk_size_tokens': 128, 'chunk_overlap_tokens': 32, 'tokenizer': 'BAAI/bge-base-en-v1.5', 'applies_to': ['fixed', 'overlapping'], 'note': 'semantic chunking size is set by threshold tuning (Phase 3.3), targeted to a comparable average so size is not confounded with strategy'}
